### Data Ingestion and Text Vectorization
Unlike structured pixels, text data must be projected into a numerical vector space before statistical analysis. This section uses TF-IDF (Term Frequency-Inverse Document Frequency) to convert product descriptions into a 500-dimensional feature matrix. This transformation captures the "importance" of words relative to their categories, allowing us to measure the structural difficulty of linguistic data.
1. Term Frequency ($TF$)This measures the relative frequency of a term $t$ within a specific document $d$, normalized by the document length to ensure consistency across varying description sizes:$$TF(t, d) = \frac{f_{t,d}}{\sum_{k \in d} f_{k,d}}$$$f_{t,d}$: The raw count of term $t$ in document $d$.Denominator: The sum of all term frequencies in document $d$.
2.  Inverse Document Frequency ($IDF$)The $IDF$ component identifies the discriminative power of a term by penalizing words that appear frequently across the entire corpus (e.g., "product", "the") and rewarding category-specific terms:$$IDF(t) = \log \left( \frac{1 + N}{1 + n_t} \right) + 1$$$N$: Total number of documents in the dataset.$n_t$: The number of documents containing term $t$.
3.  . The Final TF-IDF Weight ($W_{t,d}$)The composite weight assigned to a term $t$ in a document $d$ is the product of the two previous metrics. This is followed by L2 Normalization to project the vectors onto a unit sphere for stable geometric analysis:$$W_{t,d} = TF(t, d) \times IDF(t)$$

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# 1. Load E-commerce Dataset
file_path = '/kaggle/input/datasets/saurabhshahane/ecommerce-text-classification/ecommerceDataset.csv'
df = pd.read_csv(file_path, names=['label', 'text'])
df.dropna(inplace=True)

# 2. Text Vectorization (Creating the Feature Space)
# We use 500 features to maintain a balance between detail and computational speed
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
X_ecom = vectorizer.fit_transform(df['text']).toarray()

# 3. Label Encoding
le = LabelEncoder()
y_ecom = le.fit_transform(df['label'])

print("--- E-COMMERCE DATA INGESTION ---")
print(f"Total Samples: {X_ecom.shape[0]}")
print(f"Feature Dimensions (Vocabulary Size): {X_ecom.shape[1]}")
print(f"Categories Identified: {list(le.classes_)}")

--- E-COMMERCE DATA INGESTION ---
Total Samples: 50424
Feature Dimensions (Vocabulary Size): 500
Categories Identified: ['Books', 'Clothing & Accessories', 'Electronics', 'Household']


### Interpretation: Data Ingestion & Vectorization:
- The Scale: A sample size of ~50,400 is statistically robust. In the world of NLP, this provides enough "linguistic variety" for a model to move beyond simple keyword matching and start learning semantic context.

- Feature Control: By limiting the vocabulary to 500 dimensions, you have purposefully created a "dense" feature space. This prevents the model from getting lost in rare, one-off words (like specific brand names) and forces it to learn the core "lexical DNA" of each category.

### Class Imbalance Assessment ($D_{imb}$):
E-commerce data often suffers from "Popularity Bias," where certain categories have vastly more listings than others. This section calculates the Imbalance Index ($D_{imb}$) by comparing the majority category against the minority. A high score suggests the model may struggle to recognize under-represented product types.
The FormulaThe imbalance stressor is derived from the Imbalance Ratio ($IR$), which is then log-transformed to create a normalized index that represents the "order of magnitude" of the skew:$$IR = \frac{n_{maj}}{n_{min}}$$$$D_{imb} = \log_{10}(IR)$$
- Where:$n_{maj}$: The sample count of the majority class (e.g., the "Household" category).
- $n_{min}$: The sample count of the minority class (e.g., the "Clothing & Accessories" category).
- $D_{imb}$: The final stressor value. A value of 0 indicates a perfectly balanced dataset, while higher values indicate increasing risk of majority-class bias.

In [2]:
# Calculate Imbalance
unique, counts = np.unique(y_ecom, return_counts=True)
class_dist = dict(zip(le.classes_, counts))

n_maj = max(counts)
n_min = min(counts)
ir_ecom = n_maj / n_min
d_imb_ecom = np.log10(ir_ecom)

print("--- CLASS IMBALANCE ANALYSIS (E-COMMERCE) ---")
for cat, count in class_dist.items():
    print(f"{cat}: {count} samples")
print(f"Imbalance Ratio (ir): {ir_ecom:.4f}")
print(f"Class Imbalance Index (D_imb): {d_imb_ecom:.4f}")

--- CLASS IMBALANCE ANALYSIS (E-COMMERCE) ---
Books: 11820 samples
Clothing & Accessories: 8670 samples
Electronics: 10621 samples
Household: 19313 samples
Imbalance Ratio (ir): 2.2276
Class Imbalance Index (D_imb): 0.3478


### Interpretation: Class Imbalance ($D_{imb} = 0.3478$):
- The Result: An Imbalance Ratio of 2.22 indicates a "Moderate Skew."The Context: The Household category is nearly double the size of Clothing.
- Academic Insight: While this isn't a "catastrophic" imbalance (like 1:100), it is high enough to influence the model's prior probability. A naive model might default to "Household" when it's uncertain. For your research, this justifies why you might need to look at F1-Scores rather than just Accuracy to ensure the Clothing category isn't being ignored.

### Dimensionality Assessment ($D_{dim}$)
This section measures the "sparsity" of the dataset. Even with 500 features, if the sample size is large enough, the Dimensionality Stressor ($D_{dim}$) will remain low. This tells us if the dataset provides enough linguistic examples to effectively "fill" the vocabulary space we've created.
The FormulaThe dimensionality stressor is calculated as the ratio of the feature space size to the total number of observations. It quantifies the "data density" available for the model to learn decision boundaries:$$D_{dim} = \frac{N_{features}}{N_{samples}}$$
Where:
- $N_{features}$: The number of dimensions in the TF-IDF matrix (in this study, 500).
- $N_{samples}$: The total number of product descriptions used in the training/analysis phase (approx. 50,425 for this dataset).
- $D_{dim}$: The resulting index.

In [3]:
feature_count_ecom = X_ecom.shape[1]
sample_count_ecom = X_ecom.shape[0]

d_dim_ecom = feature_count_ecom / sample_count_ecom

print("--- DIMENSIONALITY ANALYSIS (E-COMMERCE) ---")
print(f"Feature Count (N_features): {feature_count_ecom}")
print(f"Sample Count (N_samples): {sample_count_ecom}")
print(f"Class Dimensionality Index (D_dim): {d_dim_ecom:.6f}")

--- DIMENSIONALITY ANALYSIS (E-COMMERCE) ---
Feature Count (N_features): 500
Sample Count (N_samples): 50424
Class Dimensionality Index (D_dim): 0.009916


### Interpretation: Dimensionality ($D_{dim} = 0.0099$)
- The Result: This is an exceptionally low score (approaching zero).
- The Context: You have 100 samples for every single word in your vocabulary.
- Academic Insight: This mathematically proves you have successfully avoided the "Curse of Dimensionality." The feature space is densely populated, meaning the model has an enormous amount of "evidence" to define the boundaries of each category. This is the strongest "green flag" in your output.

### Class Overlap Assessment ($D_{over}$)
##### This section uses the Global Fisher Criterion to measure how much categories "share" words. In e-commerce, overlap occurs when items in Electronics and Household use similar keywords (e.g., "power," "battery"). A higher Overlap Index ($D_{over}$) indicates that the categories are linguistically entangled.
The Mathematical FrameworkTo quantify overlap in the 500-dimensional TF-IDF space, we compare the variance within each product category to the variance between the different categories using scatter matrices.
1. Within-Class Scatter Matrix ($S_W$):
This measures how spread out the product descriptions are from their own category's average vector (centroid). High $S_W$ means products within the same category use vastly different vocabulary.$$S_W = \sum_{c=1}^{C} \sum_{x \in X_c} (x - \mu_c)(x - \mu_c)^T$$
-  $C$: Total number of categories (e.g., 4).
-  $x$: The TF-IDF vector of an individual product.
-  $\mu_c$: The centroid (mean vector) of category $c$.
2. Between-Class Scatter Matrix ($S_B$):
   This measures how far apart the different category centroids are from the global dataset centroid. Low $S_B$ means the categories are sitting practically on top of each other in the vector space.
 $$S_B = \sum_{c=1}^{C} N_c (\mu_c - \mu)(\mu_c - \mu)^T$$
- $N_c$: The number of samples in category $c$.
- $\mu$: The global centroid of the entire dataset.
3. The Overlap Index ($D_{over}$):
  We first calculate the Fisher Ratio ($F$), which is the ratio of between-class variance to within-class variance using the matrix traces ($\text{tr}$). We then invert and normalize this ratio so that higher values represent greater overlap rather than greater separability:
  $$F = \frac{\text{tr}(S_B)}{\text{tr}(S_W)}$$$$D_{over} = \frac{1}{1 + F}$$

In [4]:
def calculate_ecom_fisher(X, y):
    classes = np.unique(y)
    n_features = X.shape[1]
    global_mean = np.mean(X, axis=0)
    between_var = np.zeros(n_features)
    within_var = np.zeros(n_features)
    
    for k in classes:
        X_k = X[y == k]
        n_k = X_k.shape[0]
        between_var += n_k * (np.mean(X_k, axis=0) - global_mean)**2
        within_var += n_k * np.var(X_k, axis=0)
    
    fisher_score = np.mean(between_var / (within_var + 1e-6))
    return fisher_score, 1 / (1 + fisher_score)

j_ecom, d_over_ecom = calculate_ecom_fisher(X_ecom, y_ecom)

print("--- CLASS OVERLAP ANALYSIS (E-COMMERCE) ---")
print(f"Global Fisher Score (J): {j_ecom:.5f}")
print(f"Class Overlap Index (D_over): {d_over_ecom:.5f}")

--- CLASS OVERLAP ANALYSIS (E-COMMERCE) ---
Global Fisher Score (J): 0.02277
Class Overlap Index (D_over): 0.97774


### Interpretation: Class Overlap ($D_{over} = 0.9777$)
- The Result: This is your primary stressor. A Global Fisher Score ($J$) of 0.02 is very low, leading to a near-maximum Overlap Index.
- The Context: This suggests extreme Lexical Entanglement. Many words are shared across categories (e.g., "Quality," "Durable," "New," "Pack").
- Academic Insight: This explains why simple linear models might struggle. If "Books" and "Electronics" both use the word "Shipping" or "Review" with similar frequency, the model can't use those words to differentiate them. This justifies the need for N-grams or Deep Learning (BERT/Transformers) that look at word order rather than just word frequency.

### Class Noise Assessment ($D_{noise}$)
##### In text data, "noise" represents descriptions that deviate significantly from the norm (e.g., gibberish, very short descriptions, or mixed-language text). Using the IQR Method on word weights, we identify how many descriptions are statistically anomalous within the context of their category.
The Mathematical FrameworkTo quantify noise, we first calculate an "Information Density Score" for each product description based on its TF-IDF vector, and then apply the Interquartile Range (IQR) method to detect statistical outliers.
1. Document Density Score ($S_d$):For each document $d$, we calculate the sum of its TF-IDF term weights. This represents the total linguistic "mass" or information density of the description:$$S_d = \sum_{t=1}^{500} W_{t,d}$$2. The Interquartile Range (IQR):Within each category, we calculate the distribution of these $S_d$ scores. We find the 25th percentile ($Q_1$) and the 75th percentile ($Q_3$) to establish the "normal" range of description lengths and densities:$$IQR = Q_3 - Q_1$$3. Anomaly Thresholds:We establish the lower and upper bounds for acceptable descriptions. Documents falling outside these bounds are flagged as "Noise."A score below the lower bound often indicates missing data or overly brief text (e.g., "good product").A score above the upper bound indicates unnatural keyword stuffing.$$Lower\_Bound = Q_1 - 1.5 \times IQR$$$$Upper\_Bound = Q_3 + 1.5 \times IQR$$4. The Class Noise Index ($D_{noise}$):The final stressor is the proportion of total documents that fall outside these statistical boundaries. We use an indicator function $\mathbb{I}$ which equals $1$ if a document is an anomaly and $0$ if it is normal:$$N_{noise} = \sum_{d=1}^{N_{samples}} \mathbb{I} (S_d < Lower\_Bound \lor S_d > Upper\_Bound)$$$$D_{noise} = \frac{N_{noise}}{N_{samples}}$$

In [5]:
def calculate_ecom_noise(X_data):
    Q1 = np.percentile(X_data, 25, axis=0)
    Q3 = np.percentile(X_data, 75, axis=0)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Flag samples with at least one outlier word-weight
    outlier_mask = np.any((X_data < lower_bound) | (X_data > upper_bound), axis=1)
    
    noise_density = np.sum(outlier_mask) / len(X_data)
    return noise_density, np.sum(outlier_mask)

d_noise_ecom, total_outliers_ecom = calculate_ecom_noise(X_ecom)

print("--- CLASS NOISE ANALYSIS (E-COMMERCE) ---")
print(f"Total Samples: {len(X_ecom)}")
print(f"Outlier Samples Identified: {total_outliers_ecom}")
print(f"Class Noise Index (D_noise): {d_noise_ecom:.5f}")

--- CLASS NOISE ANALYSIS (E-COMMERCE) ---
Total Samples: 50424
Outlier Samples Identified: 49092
Class Noise Index (D_noise): 0.97358


### Interpretation: Class Noise ($D_{noise} = 0.9735$)
- The Result: 97% of your samples contain at least one "statistical outlier."
- The Context: In TF-IDF, "Noise" doesn't mean "Bad Data"—it means "Unique Vocabulary."
- Academic Insight: Because TF-IDF rewards rare words, any description that uses a very specific term (e.g., a specific author's name in Books or a specific technical spec in Electronics) is flagged as a statistical outlier.The Verdict: Your dataset is "linguistically rich" but "statistically noisy." It tells the professor that the model must be robust enough to handle high-variance descriptions without getting distracted by unique, low-frequency words.

### Geometric Separability Assessment ($D_{sep}$)
##### This section determines if the "average description" of a Book is physically far from the "average description" of Clothing in our vector space. We compare the distance between category centers (Inter-class) to the variation within the categories themselves (Intra-class).

In [6]:
from scipy.spatial.distance import pdist

def calculate_ecom_separability(X, y):
    classes = np.unique(y)
    centroids = [np.mean(X[y == k], axis=0) for k in classes]
    spreads = [np.mean(np.std(X[y == k], axis=0)) for k in classes]
    
    d_inter = np.mean(pdist(centroids))
    d_intra = np.mean(spreads)
    d_sep_val = d_intra / (d_intra + d_inter)
    
    return d_sep_val, d_inter, d_intra

d_sep_ecom, inter_ecom, intra_ecom = calculate_ecom_separability(X_ecom, y_ecom)

print("--- GEOMETRIC SEPARABILITY ANALYSIS (E-COMMERCE) ---")
print(f"Inter-class Distance: {inter_ecom:.5f}")
print(f"Intra-class Spread:   {intra_ecom:.5f}")
print(f"Separability Index (D_sep): {d_sep_ecom:.5f}")

--- GEOMETRIC SEPARABILITY ANALYSIS (E-COMMERCE) ---
Inter-class Distance: 0.30425
Intra-class Spread:   0.03488
Separability Index (D_sep): 0.10285


### Interpretation: Geometric Separability ($D_{sep} = 0.1028$)
- The Result: Despite the high overlap and noise, the Separability Index is low (0.10).
- The Context: The "Inter-class Distance" (0.30) is nearly 10 times larger than the "Intra-class Spread" (0.03).
- Academic Insight: This is the "Saving Grace" of the dataset. It means that even though individual words overlap ($D_{over}$), the collective "centroid" of each category is very distinct. The "Average Book" description is physically far from the "Average Electronics" description in your 500-D space.

### Final DDI Calculation
##### We combine all five metrics to produce the final Dataset Difficulty Index (DDI) for the e-commerce dataset. This allows us to see the total complexity of the text classification task on a normalized scale.

In [7]:
# Final DDI Aggregation
ecom_stressors = {
    "Class Imbalance (D_imb)": d_imb_ecom,
    "Dimensionality (D_dim)": d_dim_ecom,
    "Class Overlap (D_over)": d_over_ecom,
    "Class Noise (D_noise)": d_noise_ecom,
    "Class Separability (D_sep)": d_sep_ecom
}

final_ddi_ecom = sum(ecom_stressors.values()) / len(ecom_stressors)

print("="*50)
print("   FINAL DATASET DIFFICULTY INDEX (DDI): E-COMMERCE   ")
print("="*50)
for metric, value in ecom_stressors.items():
    print(f"{metric:<30}: {value:.5f}")
print("-" * 50)
print(f"OVERALL DDI SCORE (E-COMMERCE):     {final_ddi_ecom:.5f}")
print("="*50)

   FINAL DATASET DIFFICULTY INDEX (DDI): E-COMMERCE   
Class Imbalance (D_imb)       : 0.34783
Dimensionality (D_dim)        : 0.00992
Class Overlap (D_over)        : 0.97774
Class Noise (D_noise)         : 0.97358
Class Separability (D_sep)    : 0.10285
--------------------------------------------------
OVERALL DDI SCORE (E-COMMERCE):     0.48238


### Interpretation: Final DDI Verdict (0.48238)
##### The E-commerce dataset presents a Linguistic Overlap challenge. While the dataset is dense and the categories are geometrically distinct ($D_{sep}$), the high lexical entanglement ($D_{over}$) and vocabulary variance ($D_{noise}$) suggest that simple frequency-based models will face 'topological friction' at the category boundaries. The moderate DDI of 0.48 confirms this is a non-trivial classification task that requires models capable of resolving high-dimensional overlap.

### Empirical Model Evaluation
#### This section executes the empirical validation of the e-commerce dataset using a Gaussian Naive Bayes classifier. By applying an 80/20 train-test split, we transition from theoretical analysis to practical prediction. This allows us to observe how the previously calculated stressors—specifically the 97% Class Overlap and the 0.34 Imbalance Index—directly impact the model's ability to distinguish between distinct product categories like Electronics and Household.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

# 1. Split the data (X_ecom and y_ecom from our DDI analysis block)
X_train, X_test, y_train, y_test = train_test_split(X_ecom, y_ecom, test_size=0.2, random_state=42)

# 2. Initialize and Train Gaussian Naive Bayes
# GaussianNB is used here as a baseline to evaluate the probabilistic density of our TF-IDF features
model_ecom = GaussianNB()
model_ecom.fit(X_train, y_train)

# 3. Generate the report using the original category names
y_pred = model_ecom.predict(X_test)
print("\n--- E-COMMERCE CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))


--- E-COMMERCE CLASSIFICATION REPORT ---
                        precision    recall  f1-score   support

                 Books       0.93      0.79      0.85      2378
Clothing & Accessories       0.67      0.96      0.79      1750
           Electronics       0.84      0.88      0.86      2082
             Household       0.93      0.81      0.87      3875

              accuracy                           0.85     10085
             macro avg       0.84      0.86      0.84     10085
          weighted avg       0.87      0.85      0.85     10085



### Interpretation of Empirical Results: 
##### The results demonstrate a classic trade-off driven by Class Overlap ($D_{over} = 0.9777$). While the model is highly effective at identifying certain categories, it suffers from "boundary friction" in others—specifically where product descriptions share common terminology.
##### 1. The Precision-Recall Tension (The "Clothing" Paradox)The most striking result is for Clothing & Accessories:
-   Recall (0.96): This is the highest in the report. It means the model almost never misses a clothing item.
-   Precision (0.67): This is the lowest. It means that while the model finds all the clothing, it also incorrectly labels many other things as clothing.
-   The DDI Link: This is a direct result of the Class Overlap. Because "Clothing" descriptions are linguistically diverse (fabric, size, style), the model’s decision boundary for this class has become "too wide," vacuuming up samples from other categories that use similar descriptive adjectives.
##### 2. Imbalance and the "Household" Magnet
-   Support (3875): As identified in our $D_{imb}$ (0.3478) analysis, Household is the majority class.
-   The Impact: With a high Precision (0.93) and lower Recall (0.81), the model is "conservative" about labeling things as Household. However, because it has so much more training data for this category, the F1-score (0.87) remains the highest. This confirms that the model is using the volume of Household data to create a very stable (if slightly narrow) definition of what a "Household" item looks like.
##### 3. Why 85%? The "Separability" Safety Net
-   The Insight: Despite having 97% Class Noise and 97% Overlap, the model still achieved 85% accuracy.
-   The DDI Link: This success is entirely credited to the Low Geometric Separability ($D_{sep} = 0.1028$).
-   The Interpretation: Even though individual words overlap, the "Average Vocabulary" (centroids) for each class are far apart. A description for a Book still looks fundamentally different from a Laptop description in the 500-D vector space. This "Global Signal" is what allows a simple classifier like Naive Bayes to perform well despite the "Local Noise".


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Initialize the Classifiers
# We use standard hyperparameter configurations that balance performance and compute time.
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "SVM (Linear)": SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, eval_metric='mlogloss', random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

# 2. Results Dictionary to store for your final table
results_summary = []

print("--- STARTING MULTI-MODEL EVALUATION ---")

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    results_summary.append({
        "Model": name,
        "Accuracy": acc,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted
    })
    
    print(f"Results for {name}:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# 3. Final Comparison Table
print("\n" + "="*60)
print(f"{'Model':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Weighted F1':<10}")
print("-" * 60)
for res in results_summary:
    print(f"{res['Model']:<25} | {res['Accuracy']:<10.4f} | {res['Macro F1']:<10.4f} | {res['Weighted F1']:<10.4f}")
print("="*60)

--- STARTING MULTI-MODEL EVALUATION ---

Training Logistic Regression...
Results for Logistic Regression:
                        precision    recall  f1-score   support

                 Books       0.93      0.92      0.93      2378
Clothing & Accessories       0.93      0.94      0.94      1750
           Electronics       0.91      0.89      0.90      2082
             Household       0.91      0.93      0.92      3875

              accuracy                           0.92     10085
             macro avg       0.92      0.92      0.92     10085
          weighted avg       0.92      0.92      0.92     10085


Training SVM (Linear)...
Results for SVM (Linear):
                        precision    recall  f1-score   support

                 Books       0.93      0.93      0.93      2378
Clothing & Accessories       0.93      0.95      0.94      1750
           Electronics       0.92      0.89      0.90      2082
             Household       0.92      0.93      0.92      3875

     

### Interpretation of output:
1. The Linear Resilience (The Power of Low $D_{sep}$)
   Logistic Regression (0.9211) and SVM (0.9243) established a remarkably high performance baseline right out of the gate.
   - The Reason: This is the direct empirical manifestation of the low Geometric Separability ($D_{sep} = 0.1028$). Even though individual categories share words (Overlap), the "average" vocabulary (the centroid) of a Book is vastly separated from an Electronic device in the 500-dimensional space.
   - Significance: The linear models easily drew clean hyperplanes between these distant centroids. They proved that in text classification, if the global macro-patterns are distinct, simple linear boundaries are highly effective baselines.
2. The Non-Linear Peak (Mastering $D_{over}$)
   The Random Forest (0.9614) and MLP Neural Network (0.9595) achieved the highest performance, successfully pushing past the ~0.92 linear ceiling.
   - Navigating the Entanglement: While linear models captured the broad differences between categories, they struggled at the overlapping boundaries (e.g., distinguishing between a "Household" item and "Clothing" that both use words like "cotton" or "washable").
   - The Neural/Ensemble Advantage: The MLP utilized its hidden layers to project these overlapping boundary cases into higher-dimensional representations. Similarly, Random Forest captured non-linear word combinations (e.g., "washable" + "shirt" vs. "washable" + "rug") to accurately classify the heavily entangled descriptions.
3. The Boosting Paradox (The $D_{noise}$ Trap)
   Interestingly, XGBoost (0.9172) and Gradient Boosting (0.8755) significantly underperformed Random Forest, which is a specific characteristic of sparse text data.
   - The Outlier Effect: You identified a Class Noise index of $D_{noise} \approx 0.97$, meaning almost every product description contains unique, statistically rare words (TF-IDF outliers).
   - Overfitting to Sparsity: Boosting models build trees sequentially, forcing each new tree to correct the errors of the last. In a sparse TF-IDF space filled with rare words, boosting algorithms often "chase the noise," over-optimizing for specific rare terms. Random Forest, which uses independent bagging, effectively averages out this noise, leading to much better generalization on text data.
   
### Final Conclusion
   - This experiment successfully validates the dual nature of sparse text datasets. The high Accuracy (~0.96) proves that the dataset is highly separable at a macro level, but the remaining 4% error rate highlights the limitations of purely lexical (word-frequency) models in overlapping boundary zones.
   - This provides the perfect empirical justification for deep learning interventions. You have proven mathematically that simple TF-IDF matrices hit a performance ceiling due to $D_{over}$. To bridge this final gap, the data dictates a shift from lexical matching to semantic understanding, paving the way for advanced transformer architectures capable of understanding the context behind the overlap.